# CNN — Street View House Numbers (SVHN)
## Neural networks for energy applications — Day 4, Exercise 2

In this exercise we train a CNN to classify digits in the **Street View House Numbers**
(SVHN) dataset. SVHN contains real-world images of house numbers captured by Google
Street View cameras. Each image is 32x32 pixels with 3 colour channels and belongs
to one of 10 digit classes (0–9). We will also explore data augmentation techniques
and evaluate the model with a confusion matrix.

**Evaluation protocol.** We hold out 5,000 training images as a **validation set** for
monitoring, keeping the best epoch, the confusion matrix and the augmentation comparison.
The official **test** split (26,032 images) is used once, in Section 7, for the chosen model.

**Runtime.** To fit a laptop CPU, both models train on the same fixed 20,000-image subset of
the training images for 10 epochs.
This takes about 4–6 minutes on a laptop CPU and 1–2 minutes on a GPU or Apple silicon.
Increase `N_TRAIN` (up to 68,257) if you have more time; keep the budget equal for both models.

---
## 0 — Imports and Setup

In [ ]:
import copy
import os

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset

import torchvision
import torchvision.transforms as transforms

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Plotting defaults
plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 12,
})

RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# Use a GPU if there is one (CUDA, or MPS on Apple silicon), otherwise the CPU.
# GPU results can differ slightly from CPU results even with the same seed.
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

# SVHN is downloaded to ./data unless the COURSE_DATA environment variable names another folder
DATA_ROOT = os.environ.get("COURSE_DATA", "./data")
print(f"Data folder: {DATA_ROOT}")
print("Setup complete.")

---
## 1 — Load SVHN Dataset

### 1.1 Download and Prepare Data

We hold out 5,000 randomly chosen training images for validation. SVHN crops several digits
from the same house-number photograph, and the files do not record which photograph a digit
came from, so neighbouring digits of one sign can land on both sides of a random split:
validation accuracy may be slightly optimistic. The official test split is kept for the end.

In [ ]:
# Basic transforms: convert to tensor, then standardize each channel with the per-channel
# mean and std of the 73,257 SVHN training images (commonly used values). They include the
# images we hold out for validation below; leaving those out changes only the fourth decimal.
transform_basic = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4377, 0.4438, 0.4728), (0.1980, 0.2010, 0.1970)),
])

# Download SVHN
train_dataset = torchvision.datasets.SVHN(
    root=DATA_ROOT, split="train", download=True, transform=transform_basic
)
test_dataset = torchvision.datasets.SVHN(
    root=DATA_ROOT, split="test", download=True, transform=transform_basic
)

# torchvision stores the digit 0 as label 0 (the SVHN .mat files use 10), so label k = digit k
classes = [str(i) for i in range(10)]

# Fixed random split of the training images: 5,000 for validation, N_TRAIN for training
N_TRAIN = 20_000   # classroom budget (at most len(train_dataset) - 5,000 = 68,257)
perm = torch.randperm(len(train_dataset), generator=torch.Generator().manual_seed(RANDOM_STATE)).tolist()
val_indices = perm[:5000]
train_indices = perm[5000:5000 + N_TRAIN]
train_subset = Subset(train_dataset, train_indices)
val_set = Subset(train_dataset, val_indices)

print(f"Training samples   : {len(train_subset)} (of {len(train_dataset)} training images)")
print(f"Validation samples : {len(val_set)}")
print(f"Test samples       : {len(test_dataset)} (used once, in Section 7)")
print(f"Image shape        : {train_dataset[0][0].shape}")
print(f"Classes (label k = digit k): {classes}")

### 1.2 Visualize Sample Images

In [ ]:
def imshow_svhn(img):
    """Unnormalize and return numpy image."""
    img = img * torch.tensor([0.1980, 0.2010, 0.1970]).view(3, 1, 1) + \
          torch.tensor([0.4377, 0.4438, 0.4728]).view(3, 1, 1)
    img = img.clamp(0, 1)
    return np.transpose(img.numpy(), (1, 2, 0))

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, ax in enumerate(axes.flat):
    image, label = train_subset[i]
    ax.imshow(imshow_svhn(image))
    ax.set_title(f"Digit: {label}", fontsize=9)
    ax.axis("off")

plt.suptitle("SVHN Sample Images (training subset)", fontsize=14)
plt.tight_layout()
plt.show()

### 1.3 Create DataLoaders

In [ ]:
BATCH_SIZE = 128

# num_workers=0 keeps data loading in the notebook process (worker processes can hang in notebooks)
train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
# The test loader is created in Section 7, once the model choice is final

images, labels = next(iter(train_loader))
print(f"Batch image shape : {images.shape}")
print(f"Batch label shape : {labels.shape}")

---
## 2 — Build CNN for SVHN

We use a similar architecture to our CIFAR-10 model but with three convolutional
blocks and batch normalization from the start, since SVHN is a larger dataset.

Before training we pass a dummy batch through the model and print the output shape of every
convolution, pooling and linear layer: the shape ledger from the slides.

In [ ]:
class SVHNCNN(nn.Module):
    def __init__(self):
        super(SVHNCNN, self).__init__()
        # Conv Block 1: 3 -> 32
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)

        # Conv Block 2: 32 -> 64
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        # Conv Block 3: 64 -> 128
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)

        self.pool = nn.MaxPool2d(2, 2)
        self.dropout_conv = nn.Dropout2d(0.25)
        self.dropout_fc = nn.Dropout(0.5)

        # After 3 pooling layers: 32 -> 16 -> 8 -> 4
        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))  # -> 16x16
        x = self.dropout_conv(x)
        x = self.pool(F.relu(self.bn2(self.conv2(x))))  # -> 8x8
        x = self.dropout_conv(x)
        x = self.pool(F.relu(self.bn3(self.conv3(x))))  # -> 4x4
        x = self.dropout_conv(x)
        x = x.view(x.size(0), -1)
        x = self.dropout_fc(F.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

model = SVHNCNN().to(device)
print(model)

total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")

# Shape check: pass a dummy batch through the model and print the output shape of every
# convolution, pooling and linear layer (hooks run in forward order; pool is used three times)
def shape_hook(name):
    return lambda module, inputs, output: print(f"{name:<6} {tuple(output.shape)}")

hooks = [layer.register_forward_hook(shape_hook(name)) for name, layer in model.named_children()
         if isinstance(layer, (nn.Conv2d, nn.MaxPool2d, nn.Linear))]
model.eval()   # evaluation mode for the check; train_one_epoch switches back to training mode
with torch.no_grad():   # a batch of zeros: only the shapes matter, and the seeded random stream is untouched
    model(torch.zeros(8, 3, 32, 32, device=device))
for hook in hooks:
    hook.remove()

---
## 3 — Train and Evaluate

### 3.1 Training Utilities

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    return running_loss / total, correct / total


def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    return running_loss / total, correct / total

print("Training and evaluation functions defined.")

### 3.2 Train for 10 Epochs

After every epoch we evaluate on the **validation** set and keep a copy of the weights from
the epoch with the highest validation accuracy.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

NUM_EPOCHS = 10   # classroom budget; with more time, raise it together with N_TRAIN
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc, best_epoch, best_state = 0.0, 0, None

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    if val_acc > best_val_acc:
        best_val_acc, best_epoch = val_acc, epoch
        best_state = copy.deepcopy(model.state_dict())

    print(f"Epoch {epoch:>2d}/{NUM_EPOCHS}  "
          f"Train Loss: {train_loss:.4f}  Train Acc: {train_acc:.4f}  "
          f"Val Loss: {val_loss:.4f}  Val Acc: {val_acc:.4f}")

model.load_state_dict(best_state)
print(f"\nKeeping the weights from epoch {best_epoch} (validation accuracy {best_val_acc:.4f})")

---
## 4 — Plot Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, NUM_EPOCHS + 1)

ax1.plot(epochs, history["train_loss"], "o-", label="Train Loss", color="steelblue")
ax1.plot(epochs, history["val_loss"], "s--", label="Validation Loss", color="tomato")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Loss Curves")
ax1.legend()

ax2.plot(epochs, history["train_acc"], "o-", label="Train Accuracy", color="steelblue")
ax2.plot(epochs, history["val_acc"], "s--", label="Validation Accuracy", color="tomato")
ax2.axvline(best_epoch, color="gray", linestyle=":", label=f"Kept epoch ({best_epoch})")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_title("Accuracy Curves")
ax2.legend()

plt.suptitle("SVHN CNN Training Curves", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 5 — Confusion Matrix (validation set)

In [ ]:
all_preds = []
all_labels = []

model.eval()
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
disp.plot(ax=ax, cmap="Blues", values_format="d")
ax.set_title("SVHN Confusion Matrix (validation set)", fontsize=14)
ax.grid(False)
plt.tight_layout()
plt.show()

# Per-class accuracy from confusion matrix
per_class_acc = cm.diagonal() / cm.sum(axis=1)
for i, acc in enumerate(per_class_acc):
    print(f"Digit {i}: {acc:.4f}")

---
## 6 — Data Augmentation

We train a fresh model on the **same training images** with random augmentations and
compare it with the first model on the **validation** set, with the same epoch budget.

- **Small random crops** (pad by 2 pixels, crop back to 32 × 32): the digit shifts by up to
  2 pixels. SVHN crops often show neighbouring digits at the sides and the label is the digit in
  the centre, so we keep the shifts small enough that the labelled digit stays in the centre.
- **Colour jitter** (brightness, contrast): house numbers appear in many colours and lighting
  conditions.
- **No horizontal flip.** Most digits are not left-right symmetric: a mirrored 2, 3, 4, 5, 6, 7
  or 9 is no longer that digit, so a flip would train the model on images whose label no longer
  matches what they show. An augmentation must preserve the label.

Augmentation is applied to training images only. Validation and test images keep the
deterministic `transform_basic`. Augmented images are harder to fit, so with a short training
budget augmentation can lower validation accuracy even where it would help with longer training:
judge it on validation data rather than assuming a gain.

In [ ]:
transform_augmented = transforms.Compose([
    transforms.RandomCrop(32, padding=2),          # shifts of up to 2 pixels keep the labelled digit centred
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.4377, 0.4438, 0.4728), (0.1980, 0.2010, 0.1970)),
])

# Same training images as before (train_indices), now with random augmentation
train_dataset_aug = torchvision.datasets.SVHN(
    root=DATA_ROOT, split="train", download=True, transform=transform_augmented
)
train_loader_aug = DataLoader(Subset(train_dataset_aug, train_indices), batch_size=BATCH_SIZE,
                              shuffle=True, num_workers=0)

# New model instance for fair comparison
torch.manual_seed(RANDOM_STATE)
model_aug = SVHNCNN().to(device)
criterion_aug = nn.CrossEntropyLoss()
optimizer_aug = optim.Adam(model_aug.parameters(), lr=1e-3)

history_aug = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc_aug, best_epoch_aug, best_state_aug = 0.0, 0, None

for epoch in range(1, NUM_EPOCHS + 1):   # same budget as the first model
    train_loss, train_acc = train_one_epoch(model_aug, train_loader_aug, criterion_aug, optimizer_aug, device)
    val_loss, val_acc = evaluate(model_aug, val_loader, criterion_aug, device)

    history_aug["train_loss"].append(train_loss)
    history_aug["train_acc"].append(train_acc)
    history_aug["val_loss"].append(val_loss)
    history_aug["val_acc"].append(val_acc)
    if val_acc > best_val_acc_aug:
        best_val_acc_aug, best_epoch_aug = val_acc, epoch
        best_state_aug = copy.deepcopy(model_aug.state_dict())

    print(f"Epoch {epoch:>2d}/{NUM_EPOCHS}  "
          f"Train Loss: {train_loss:.4f}  Train Acc: {train_acc:.4f}  "
          f"Val Loss: {val_loss:.4f}  Val Acc: {val_acc:.4f}")

model_aug.load_state_dict(best_state_aug)
print(f"\nKeeping the weights from epoch {best_epoch_aug} (validation accuracy {best_val_acc_aug:.4f})")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, NUM_EPOCHS + 1)

ax1.plot(epochs_range, history["val_acc"], "s--", label="No Augmentation", color="tomato")
ax1.plot(epochs_range, history_aug["val_acc"], "o-", label="With Augmentation", color="steelblue")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Validation Accuracy")
ax1.set_title("Validation Accuracy: Augmentation Comparison")
ax1.legend()

ax2.plot(epochs_range, history["val_loss"], "s--", label="No Augmentation", color="tomato")
ax2.plot(epochs_range, history_aug["val_loss"], "o-", label="With Augmentation", color="steelblue")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Validation Loss")
ax2.set_title("Validation Loss: Augmentation Comparison")
ax2.legend()

plt.suptitle("Effect of Data Augmentation on SVHN (validation set)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f"Without augmentation best validation acc: {best_val_acc:.4f} (epoch {best_epoch})")
print(f"With augmentation best validation acc   : {best_val_acc_aug:.4f} (epoch {best_epoch_aug})")

if best_val_acc_aug > best_val_acc:
    final_model, final_name = model_aug, "with augmentation"
else:
    final_model, final_name = model, "without augmentation"
print(f"Chosen on validation data: the model {final_name}")

---
## 7 — Final Test Evaluation (once)

The choice is now locked: the model with the higher validation accuracy, with the weights of
its best validation epoch. We evaluate it once on the official test split. Changing the model
after seeing this number would turn the test set into another validation set.

In [ ]:
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

test_loss, test_acc = evaluate(final_model, test_loader, criterion, device)
print(f"Final model         : SVHNCNN {final_name}")
print(f"Final Test Loss     : {test_loss:.4f}")
print(f"Final Test Accuracy : {test_acc:.4f}")

---
## Summary

In this exercise we:

1. **Loaded and explored** the SVHN dataset (32x32 colour images of house number digits) and
   held out 5,000 training images for validation.
2. **Built a CNN** with three convolutional blocks, batch normalization, and dropout, and
   printed the output shape of every convolution, pooling and linear layer.
3. **Trained** the model for 10 epochs on a 20,000-image subset, plotted training and
   validation curves and kept the best validation epoch.
4. **Computed a confusion matrix** on the validation set to see which digits are confused.
5. **Applied label-preserving data augmentation** (small random crops and colour jitter, no
   horizontal flip) and compared it with the first model on the validation set.
6. **Evaluated the chosen model once** on the test set.

Key takeaways:
- SVHN is larger than CIFAR-10, and its street-level crops vary widely in quality and often
  show neighbouring digits.
- The confusion matrix reveals which digit pairs are commonly misclassified.
- An augmentation must preserve the label: shifts and colour changes keep a digit's identity,
  mirroring does not.
- Compare augmentation choices on validation data with the same budget; use the test set once.